# 01 · Sparse reconstruction of the radiator

Phone video → sharp frames → structure-from-motion (pycolmap, CPU) → interactive 3D view.

Each step has a **definitions** cell (functions) followed by a **run** cell (calls + plots). Once a step stabilises, its definitions cell moves into `src/` as a module.

In [ ]:
from __future__ import annotations

import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import pycolmap

pio.renderers.default = "notebook_connected"   # "vscode" if running inside VS Code

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VIDEO = ROOT / "data/raw/VID20261023102419.mp4"   # clip 3: the clean arc
WORK = ROOT / "outputs/clip3"
GT = yaml.safe_load(open(ROOT / "data/ground_truth/radiator_2026-10-23.yaml"))
GT["measured"]

## 1 · Frames and blur filter
Extract at 4 fps, then keep the sharpest 75 % (variance of the Laplacian).

In [ ]:
# --- definitions: frames ---
def extract_frames(video: Path, out_dir: Path, fps: float = 4.0) -> list[Path]:
    """Extract frames with ffmpeg (respects the phone's rotation metadata)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["ffmpeg", "-v", "error", "-y", "-i", str(video),
         "-vf", f"fps={fps}", "-q:v", "2", str(out_dir / "%04d.jpg")],
        check=True,
    )
    return sorted(out_dir.glob("*.jpg"))


def sharpness(path: Path) -> float:
    """Variance of the Laplacian: low = blurry."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return float(cv2.Laplacian(img, cv2.CV_64F).var())


def filter_sharp(frames: list[Path], out_dir: Path, keep_fraction: float = 0.75) -> tuple[list[Path], np.ndarray]:
    """Copy the sharpest `keep_fraction` of frames to out_dir. Returns kept paths and all scores."""
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    scores = np.array([sharpness(f) for f in frames])
    threshold = np.percentile(scores, 100 * (1 - keep_fraction))
    kept = []
    for f, s in zip(frames, scores):
        if s >= threshold:
            kept.append(Path(shutil.copy(f, out_dir)))
    return kept, scores


In [ ]:
# --- run: frames ---
all_frames = extract_frames(VIDEO, WORK / "raw", fps=4)
kept, scores = filter_sharp(all_frames, WORK / "frames", keep_fraction=0.75)
thr = np.percentile(scores, 25)

fig = go.Figure(go.Scatter(y=scores, mode="lines+markers", marker=dict(size=4,
        color=np.where(scores >= thr, "#1D9E75", "#E24B4A")), name="sharpness"))
fig.add_hline(y=thr, line_dash="dash", annotation_text="keep above")
fig.update_layout(height=280, xaxis_title="frame", yaxis_title="Laplacian variance", margin=dict(t=20))
fig.show()
print(f"{len(all_frames)} extracted, {len(kept)} kept")

In [ ]:
# Sharpest vs blurriest frames
order = np.argsort(scores)
fig, ax = plt.subplots(2, 4, figsize=(12, 7))
for a, i in zip(ax[0], order[-4:][::-1]): a.imshow(plt.imread(all_frames[i])); a.set_title(f"sharp {scores[i]:.0f}")
for a, i in zip(ax[1], order[:4]):        a.imshow(plt.imread(all_frames[i])); a.set_title(f"blurry {scores[i]:.0f}")
for a in ax.flat: a.axis("off")
plt.tight_layout()

## 2 · Structure-from-motion
SIFT → sequential matching → incremental mapping. About 1 min on a laptop CPU; cached in `outputs/` afterwards (`overwrite=True` to rerun).

In [ ]:
# --- definitions: SfM ---
def run_sfm(image_dir: Path, out_dir: Path, max_features: int = 4000, overlap: int = 12,
            overwrite: bool = False) -> pycolmap.Reconstruction:
    """SIFT features -> sequential matching -> incremental mapping. Returns the largest model.

    Results are cached in out_dir; pass overwrite=True to recompute.
    """
    image_dir, out_dir = Path(image_dir), Path(out_dir)
    model_dir = out_dir / "0"
    if model_dir.exists() and not overwrite:
        return pycolmap.Reconstruction(model_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    db = out_dir / "database.db"

    extraction = pycolmap.FeatureExtractionOptions()
    extraction.sift.max_num_features = max_features
    pycolmap.extract_features(db, image_dir, camera_mode=pycolmap.CameraMode.SINGLE,
                              extraction_options=extraction, device=pycolmap.Device.cpu)

    pairing = pycolmap.SequentialPairingOptions()
    pairing.overlap = overlap
    pairing.loop_detection = False
    pycolmap.match_sequential(db, pairing_options=pairing, device=pycolmap.Device.cpu)

    models = pycolmap.incremental_mapping(db, image_dir, out_dir)
    if not models:
        raise RuntimeError("No model could be reconstructed")
    best = max(models, key=lambda k: models[k].num_reg_images())
    if best != 0:  # keep the largest model at out_dir/0 for caching
        shutil.rmtree(model_dir, ignore_errors=True)
        (out_dir / str(best)).rename(model_dir)
    return pycolmap.Reconstruction(model_dir)


@dataclass
class Cloud:
    xyz: np.ndarray        # (N, 3) points
    rgb: np.ndarray        # (N, 3) uint8 colours
    error: np.ndarray      # (N,) reprojection error, px
    track: np.ndarray      # (N,) number of images observing each point
    cam_centers: np.ndarray  # (M, 3) camera centres, in frame order
    cam_rotations: np.ndarray  # (M, 3, 3) world-to-camera rotations
    names: list[str]


def to_cloud(rec: pycolmap.Reconstruction) -> Cloud:
    pts = list(rec.points3D.values())
    ims = sorted(rec.images.values(), key=lambda im: im.name)
    return Cloud(
        xyz=np.array([p.xyz for p in pts]),
        rgb=np.array([p.color for p in pts], dtype=np.uint8),
        error=np.array([p.error for p in pts]),
        track=np.array([p.track.length() for p in pts]),
        cam_centers=np.array([im.projection_center() for im in ims]),
        cam_rotations=np.array([im.cam_from_world().rotation.matrix() for im in ims]),
        names=[im.name for im in ims],
    )


def clean(cloud: Cloud, max_error: float = 1.5, min_track: int = 3, keep_percentile: float = 97) -> Cloud:
    """Drop high-error / short-track points and far outliers."""
    m = (cloud.error < max_error) & (cloud.track >= min_track)
    xyz = cloud.xyz[m]
    d = np.linalg.norm(xyz - np.median(xyz, axis=0), axis=1)
    m2 = d < np.percentile(d, keep_percentile)
    idx = np.flatnonzero(m)[m2]
    return Cloud(cloud.xyz[idx], cloud.rgb[idx], cloud.error[idx], cloud.track[idx],
                 cloud.cam_centers, cloud.cam_rotations, cloud.names)


def summary(rec: pycolmap.Reconstruction, n_input: int) -> dict:
    return {
        "input_frames": n_input,
        "registered": rec.num_reg_images(),
        "registered_pct": round(100 * rec.num_reg_images() / n_input, 1),
        "points": rec.num_points3D(),
        "mean_reproj_error_px": round(rec.compute_mean_reprojection_error(), 3),
        "mean_track_length": round(rec.compute_mean_track_length(), 2),
    }


In [ ]:
# --- run: SfM ---
rec = run_sfm(WORK / "frames", WORK / "sparse")
summary(rec, n_input=len(kept))

## 3 · Look at it
Red pyramids = where the phone was for each registered frame. Drag to rotate.

In [ ]:
# --- definitions: visualisation ---
def gravity_frame(cloud: Cloud) -> np.ndarray:
    """Rotation that makes 'up' = average image-up direction of the cameras.

    Assumes the phone was held roughly upright. Returns R so that R @ x maps world -> (right, up, back).
    """
    R = cloud.cam_rotations
    up = -np.mean([r.T @ np.array([0, 1, 0]) for r in R], axis=0)
    up /= np.linalg.norm(up)
    fwd = np.mean([r.T @ np.array([0, 0, 1]) for r in R], axis=0)
    fwd -= up * (fwd @ up)
    fwd /= np.linalg.norm(fwd)
    return np.stack([np.cross(fwd, up), up, -fwd])


def frustum_lines(cloud: Cloud, size: float) -> np.ndarray:
    """Line segments (NaN-separated) drawing a small pyramid per camera."""
    segs = []
    corners = np.array([(-.56, -1, 1.2), (.56, -1, 1.2), (.56, 1, 1.2), (-.56, 1, 1.2)]) * size
    for R, c in zip(cloud.cam_rotations, cloud.cam_centers):
        k = c + corners @ R  # camera -> world
        for a, b in [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (2, 3), (3, 4), (4, 1)]:
            p = np.vstack([c, k])
            segs += [p[a], p[b], [np.nan] * 3]
    return np.array(segs)


def plot_cloud(cloud: Cloud, show_cameras: bool = True, point_size: float = 2.0,
               extra: list[go.BaseTraceType] | None = None, title: str = "") -> go.Figure:
    R = gravity_frame(cloud)
    mu = cloud.xyz.mean(axis=0)
    tf = lambda a: (a - mu) @ R.T  # noqa: E731
    p = tf(cloud.xyz)
    # plotly: x right, y depth, z up
    traces = [go.Scatter3d(x=p[:, 0], y=-p[:, 2], z=p[:, 1], mode="markers", name="points",
                           marker=dict(size=point_size, color=[f"rgb({r},{g},{b})" for r, g, b in cloud.rgb.tolist()]))]
    if show_cameras:
        scale = 0.03 * np.ptp(p, axis=0).max()
        f = frustum_lines(cloud, scale)
        f = np.where(np.isnan(f), np.nan, tf(np.nan_to_num(f)))
        traces.append(go.Scatter3d(x=f[:, 0], y=-f[:, 2], z=f[:, 1], mode="lines", name="cameras",
                                   line=dict(color="#E24B4A", width=2)))
    traces += extra or []
    fig = go.Figure(traces)
    fig.update_layout(title=title, height=650, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
    return fig


In [ ]:
# --- run: visualisation ---
cloud = clean(to_cloud(rec))
plot_cloud(cloud, title=f"Clip 3 · {len(cloud.xyz)} points, {len(cloud.cam_centers)} cameras").show()

## 4 · Where are the points? (texture check)
Points come from texture, so the white tube surfaces should be sparse compared with edges, floor and skirting board (see `docs/FINDINGS.md`). Points observed per frame:

In [ ]:
obs = sorted((im.name, im.num_points3D) for im in rec.images.values())
fig = go.Figure(go.Bar(x=[n for n, _ in obs], y=[k for _, k in obs]))
fig.update_layout(height=280, xaxis_title="frame", yaxis_title="3D points seen", margin=dict(t=20))
fig.show()

## Your turn
Each is a one-line change in a run cell:
- `keep_fraction` 0.5 vs 0.9: does a stricter blur filter register more or fewer frames?
- `fps=2` vs `fps=6`: fewer, more different frames vs more overlap.
- `overlap=5` vs `overlap=25` in `run_sfm` (use `overwrite=True`): speed vs robustness.

Log what you find in `docs/FINDINGS.md`. **Next notebook (02):** metric scale from the bank card, then compare against `GT`.